# DeepLog (LSTM) sur HDFS_v1

**Principe.** DeepLog apprend **l’ordre habituel des événements** dans les blocs normaux. Ensuite, pour chaque bloc, il essaie de deviner quel événement va arriver ensuite. S’il est souvent surpris, c’est que le bloc ne suit pas l’ordre habituel : on le signale comme anormal

## 0. Imports et configuration

- `SEED = 42` : résultats reproductibles (règle 5).
- `NUM_EVENTS = 30` : types d'événements E1 à E30.
- `MAX_WINDOWS_PER_BLOCK = 30` : au plus 30 fenêtres par bloc à l'entraînement, pour aller plus vite.
- `DEVICE` : GPU s'il existe, sinon CPU.

In [36]:
import random
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

sys.path.append("../..")  # "../.." = hdfs_v1/
from donnees import charger_sequences, ecrire_predictions, parties, seuil

SEED = 42
NUM_EVENTS = 30
MAX_WINDOWS_PER_BLOCK = None
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print("Device :", DEVICE)

Device : cpu


## 1. Charger `sequences.parquet`

`charger_sequences()` renvoie un tableau avec une ligne par bloc : `block_id`, `label`, `split` et `events` (la liste des événements dans l'ordre du temps). `parties(df)` découpe ce tableau en groupes (`train_normal`, `val`, `val_normal`, `test`...).

In [37]:
df = charger_sequences()
p = parties(df)
df.head()

,block_id,label,split,events
0,blk_-1608999687919862906,0,train,"[5, 22, 5, 5, 11, 11, 9, 9, 11, 9, 26, 26, 26,..."
1,blk_7503483334202473044,0,train,"[5, 5, 22, 5, 11, 9, 11, 9, 11, 9, 26, 26, 26,..."
2,blk_-3544583377289625738,1,train,"[5, 22, 5, 5, 11, 9, 11, 9, 11, 9, 3, 26, 26, ..."
3,blk_-9073992586687739851,0,train,"[5, 22, 5, 5, 11, 9, 11, 9, 11, 9, 26, 26, 26,..."
4,blk_7854771516489510256,0,train,"[5, 5, 22, 5, 11, 9, 11, 9, 11, 9, 26, 26, 26,..."


## 2. Choisir les données de chaque étape

| Variable | Groupe | Rôle |
|---|---|---|
| `train_sequences` | `train_normal` | **apprendre** (blocs normaux uniquement) |
| `val_sequences` | `val` | calculer le F1 et choisir le réglage |
| `val_normal_sequences` | `val_normal` | calculer le seuil d'alerte |
| `test_sequences` | `test` | note finale, **une seule fois** |

In [38]:

val_sequences = p["val"]["events"].tolist()
val_normal_sequences = p["val_normal"]["events"].tolist()
test_sequences = p["test"]["events"].tolist()
train_sequences = sorted(list(s) for s in set(tuple(s) for s in p["train_normal"]["events"]))

print("train (séquences uniques) :", len(train_sequences))
print("train_normal :", len(train_sequences))
print("val          :", len(val_sequences))
print("val_normal   :", len(val_normal_sequences))
print("test         :", len(test_sequences))

train (séquences uniques) : 13751
train_normal : 13751
val          : 86259
val_normal   : 83652
test         : 86260


## 3. Transformer les séquences en fenêtres glissantes

Un LSTM apprend à partir d'exemples « entrée → sortie ». Pour `[5, 5, 22, 11, 9, 11]` avec `window_size=3` :

| Entrée (fenêtre) | Cible (événement suivant) |
|---|---|
| `[5, 5, 22]` | 11 |
| `[5, 22, 11]` | 9 |
| `[22, 11, 9]` | 11 |

- Les blocs de longueur ≤ `window_size` sont ignorés (pas de cible).
- On retire 1 à la cible car les événements vont de 1 à 30 alors que les classes PyTorch vont de 0 à 29.
- `make_loader` empile les fenêtres dans un `DataLoader` (lots de 256, mélangés).

In [39]:
END = 30  

def augmenter(seq, window_size):
    return [0] * window_size + list(seq) + [END]

def make_windows(sequences, window_size, max_windows_per_block=None):
    inputs, targets = [], []
    for sequence in sequences:
        s2 = augmenter(sequence, window_size)
        starts = range(len(s2) - window_size)
        if max_windows_per_block is not None:
            starts = list(starts)[:max_windows_per_block]
        for start in starts:
            inputs.append(s2[start:start + window_size])
            targets.append(s2[start + window_size] - 1)
    return torch.tensor(inputs, dtype=torch.long), torch.tensor(targets, dtype=torch.long)

def make_loader(sequences, window_size, batch_size=256):
    x, y = make_windows(sequences, window_size, MAX_WINDOWS_PER_BLOCK)
    return DataLoader(TensorDataset(x, y), batch_size=batch_size, shuffle=True)

## 4. Le modèle : un LSTM qui prédit l'événement suivant

Trois couches :
1. **Embedding** : chaque numéro d'événement devient un vecteur de taille `hidden_size` (comme un mot dans une phrase).
2. **LSTM** : lit la fenêtre dans l'ordre.
3. **Linear** : à partir de la dernière sortie du LSTM, donne un score pour chacun des 30 événements possibles.

`train_model` entraîne avec **Adam** (`lr=1e-3`) et **CrossEntropyLoss** : le modèle doit donner la meilleure probabilité à l'événement réellement suivant.

In [40]:
class DeepLogLSTM(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(NUM_EVENTS, hidden_size)
        self.lstm = nn.LSTM(hidden_size, hidden_size, batch_first=True)
        self.output = nn.Linear(hidden_size, NUM_EVENTS)

    def forward(self, x):
        embedded = self.embedding(x)
        output, _ = self.lstm(embedded)
        return self.output(output[:, -1, :])

def train_model(sequences, window_size, hidden_size, epochs, top_k=None, batch_size=256):
    torch.manual_seed(SEED)
    model = DeepLogLSTM(hidden_size).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    loader = make_loader(sequences, window_size, batch_size)
    model.train()
    for _ in range(epochs):
        for x_batch, y_batch in loader:
            x_batch, y_batch = x_batch.to(DEVICE), y_batch.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(x_batch), y_batch)
            loss.backward()
            optimizer.step()
    return model

## 5. Score d'anomalie : les « surprises » (top-k)

Pour chaque bloc :
1. on crée toutes ses fenêtres ;
2. le modèle prédit les `top_k` événements les plus probables ;
3. une fenêtre est une **surprise** si l'événement réel n'est pas dans ce top-k ;
4. le **score du bloc = nombre de surprises** (plus haut = plus anormal).

`@torch.no_grad()` désactive les gradients : plus rapide, car on n'entraîne pas ici.


In [41]:
def make_windows_with_owner(sequences, window_size):
    inputs, targets, owners = [], [], []
    for i, seq in enumerate(sequences):
        s2 = augmenter(seq, window_size)
        for s in range(len(s2) - window_size):
            inputs.append(s2[s:s + window_size])
            targets.append(s2[s + window_size] - 1)
            owners.append(i)
    return (torch.tensor(inputs, dtype=torch.long),
            torch.tensor(targets, dtype=torch.long),
            np.asarray(owners))


@torch.no_grad()
def sequence_scores(model, sequences, window_size, top_k, batch_size=8192):
    model.eval()
    x, y, owners = make_windows_with_owner(sequences, window_size)
    surprise = []
    for i in range(0, len(x), batch_size):
        xb = x[i:i + batch_size].to(DEVICE)
        yb = y[i:i + batch_size].to(DEVICE)
        top = model(xb).topk(top_k, dim=1).indices
        surprise.append((top != yb.unsqueeze(1)).all(dim=1).cpu())
    surprise = torch.cat(surprise).numpy()
    return np.bincount(owners[surprise], minlength=len(sequences)).astype(float)

## 6. Tester plusieurs réglages sur `val`

Pour chaque configuration (`window_size`, `hidden_size`, `top_k`, `epochs`) :
- on entraîne sur `train_normal` ;
- on calcule le seuil avec `seuil()` sur les scores de `val_normal` (quantile 99,5 %) ;
- on met en alerte les blocs de `val` dont le score dépasse ce seuil, puis on calcule le **F1**.

Le test n'est pas touché ici.

In [44]:
from sklearn.metrics import precision_score, recall_score

config = {"window_size": 8, "hidden_size": 64, "epochs": 3, "batch_size": 1024}
model = train_model(train_sequences, **config)
y = p["val"]["label"].values

validation_results = []
for top_k in [1, 2, 3, 4]:
    val_scores = sequence_scores(model, val_sequences, config["window_size"], top_k)
    validation_threshold = seuil(val_scores[y == 0])
    val_pred = (val_scores > validation_threshold).astype(int)
    f1 = f1_score(y, val_pred, zero_division=0)
    validation_results.append({**config, "top_k": top_k,
                               "threshold": validation_threshold, "f1": f1})
    print("top_k =", top_k, "| seuil =", validation_threshold,
          "| alertes =", int(val_pred.sum()),
          "| précision =", round(precision_score(y, val_pred, zero_division=0), 3),
          "| rappel =", round(recall_score(y, val_pred, zero_division=0), 3),
          "| F1 =", round(f1, 4))

top_k = 1 | seuil = 6.0 | alertes = 780 | précision = 0.583 | rappel = 0.175 | F1 = 0.2687
top_k = 2 | seuil = 2.0 | alertes = 795 | précision = 0.647 | rappel = 0.197 | F1 = 0.3022
top_k = 3 | seuil = 1.0 | alertes = 1574 | précision = 0.742 | rappel = 0.448 | F1 = 0.5587
top_k = 4 | seuil = 1.0 | alertes = 1093 | précision = 0.79 | rappel = 0.331 | F1 = 0.467


## 7. Choisir le meilleur réglage (F1 sur `val`)

On trie les configurations par F1 décroissant et on garde la première.

In [ ]:
results_df = pd.DataFrame(validation_results).sort_values("f1", ascending=False)
best_config = results_df.iloc[0].to_dict()
best_config = {
    key: int(best_config[key])
    for key in ("window_size", "hidden_size", "top_k", "epochs")
}
print(results_df)
print("Meilleur reglage:", best_config)

   window_size  hidden_size  top_k  epochs  batch_size  threshold        f1
1            5           64      6       5        1024        7.0  0.415995
2            8           64      4       5        1024        7.0  0.415476
4            8           64      7       5        1024        7.0  0.415320
3            8           64      6       5        1024        7.0  0.415196
0            5           64      4       5        1024        7.0  0.415106
Meilleur reglage: {'window_size': 5, 'hidden_size': 64, 'top_k': 6, 'epochs': 5}


## 9. Test (une seule fois) et fichier de prédictions

On note les blocs de `test` **une seule fois**, on mesure le temps de prédiction (`--pred-time`) et on écrit `results/deeplog_test.csv`.

In [ ]:
prediction_started = time.perf_counter()
test_scores = sequence_scores(
    best_model, test_sequences, best_config["window_size"], best_config["top_k"]
)
prediction_time_s = time.perf_counter() - prediction_started
output_path = Path("../../results/deeplog_v2_test.csv")
ecrire_predictions(p["test"], test_scores, final_threshold, output_path)
print("Predictions ecrites dans:", output_path)
print("Temps entrainement (min):", round(train_time_min, 2))
print("Temps prediction (s):", round(prediction_time_s, 2))

Predictions ecrites dans: ..\..\results\deeplog_test.csv
Temps entrainement (min): 0.37
Temps prediction (s): 22.45


## Évaluation des performances du modèle DeepLog

In [47]:

from sklearn.metrics import precision_score, recall_score, f1_score

# 1. Vitesse d'entraînement
train_time_s = train_time_min * 60

# Nombre de fenêtres utilisées pendant l'entraînement
nb_train_windows = sum(len(seq) + 1 for seq in train_sequences)

train_throughput = (
    nb_train_windows * best_config["epochs"] / train_time_s
)

# 2. Vitesse de réponse
nb_test_blocks = len(test_sequences)

response_per_block_ms = (
    prediction_time_s / nb_test_blocks
) * 1000

response_throughput = nb_test_blocks / prediction_time_s

# 3. Métriques de détection sur le jeu de test
y_test = p["test"]["label"].values

test_predictions = (test_scores > final_threshold).astype(int)

precision = precision_score(y_test, test_predictions, zero_division=0)
recall = recall_score(y_test, test_predictions, zero_division=0)
f1 = f1_score(y_test, test_predictions, zero_division=0)

# 4. Affichage des résultats
print("========== PERFORMANCE DEEPLOG ==========")

print("\n--- Vitesse d'entraînement ---")
print(f"Temps d'entraînement : {train_time_s:.2f} secondes")


print("\n--- Vitesse de réponse ---")
print(f"Temps total : {prediction_time_s:.4f} secondes")

print("\n--- Qualité de détection ---")
print(f"Précision : {precision:.4f}")
print(f"Rappel (Recall)   : {recall:.4f}")
print(f"F1-score  : {f1:.4f}")

print("=========================================")


========== PERFORMANCE DEEPLOG ==========

--- Vitesse d'entraînement ---
Temps d'entraînement : 22.01 secondes

--- Vitesse de réponse ---
Temps total : 22.4524 secondes

--- Qualité de détection ---
Précision : 0.9035
Rappel (Recall)   : 0.1846
F1-score  : 0.3065
